In [1]:
import os

In [2]:
%pwd

'c:\\Users\\User\\Desktop\\DS\\Kidney_cancer\\research'

In [3]:
os.chdir("..") 

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class BaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [5]:
from kidney_cancer.constants import  *
from kidney_cancer.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_base_model_config(self) -> BaseModelConfig:
        config = self.config.prepare_base_model
        
        create_directories([config.root_dir])

        base_model_config = BaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_classes=self.params.CLASSES,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS 
        )

        return base_model_config

In [7]:
import os
import urllib.request as request
from zipfile import ZipFile
from torchvision import models
import torchvision
import torch
from torch import nn

In [8]:
class PrepareBaseModel:
    def __init__(self, config: BaseModelConfig):
        self.config = config

    @staticmethod
    def save_model(path: Path, model: torchvision.models):
        torch.save(model, path)

    def get_base_model(self):
        self.model = models.vgg16(weights=self.config.params_weights)
        self.save_model(path=self.config.base_model_path, model=self.model)

    @staticmethod
    def _prepare_full_model(model, classes, freeze_features, freeze_till=None):
        if freeze_features:
            for param in model.features.parameters():   
                param.requires_grad = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for param in list(model.parameters())[:-freeze_till]:
                param.requires_grad = False
        
        model.classifier[6] = nn.Linear(in_features=model.classifier[6].in_features, out_features=classes)

        print(model)
        return model
    
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_features=True
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

In [19]:
model = torch.load("model/model.pth", weights_only=False)

In [31]:
import torchvision.io as io
img_path = r"artifacts\data_ingestion\Kidney_dataset\Train\Normal\Normal- (2272).jpg"
img = io.read_image(img_path)

In [32]:
from torchvision.transforms import v2
transforms = v2.Compose([
v2.ToImage(),
v2.Resize(size=[224,224]),
v2.ToDtype(dtype=torch.float32,scale=True)])

In [55]:
y_pred = torch.sigmoid(model(transforms(img.unsqueeze(0))))

In [50]:
y_pred = y_pred.detach().numpy()

np.float32(1.0)

In [ ]:
labels = ["Normal","Tumor"]


{'Normal': 1.0, 'Tumor': 0.0}

In [46]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2025-07-26 17:07:57,060: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-07-26 17:07:57,064: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-26 17:07:57,066: INFO: common: created directory at: artifacts]
[2025-07-26 17:07:57,067: INFO: common: created directory at: artifacts/base_model]
VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(